# NSE Swing Trading Scanner
**Run cells 1 → 2 → 3 once to load everything.**  
Then use **Cell 4** for manual runs, or **Cell 5 + Cell 6** for the daily 4 PM auto-scheduler.

| Cell | What it does |
|---|---|
| 1 | Install & imports |
| 2 | Configuration (edit your capital, risk %) |
| 3 | Full scanner engine |
| 4 | Manual run |
| 5 | Daily 4 PM scheduler |
| 6 | Keep-alive JS (prevents Colab timeout) |


## Cell 0


In [ ]:
# ============================================================
# NSE SWING TRADING SCANNER — FULL PRODUCTION NOTEBOOK
# Author  : Quantitative Trader Build
# Version : 1.0
# Run     : Automatically at 16:00 IST daily | Manual anytime
# ============================================================
# COLAB USAGE:
#   Cell 1  → Install & Imports
#   Cell 2  → Configuration (edit this)
#   Cell 3  → Scanner Engine (do not edit)
#   Cell 4  → Manual Run: run_scanner()
#   Cell 5  → Auto Scheduler (run once, leave open)
#   Cell 6  → Keep-Alive JS (run once alongside Cell 5)
# ============================================================

## CELL 1 — INSTALL & IMPORTS


In [ ]:
# ║  CELL 1 — INSTALL & IMPORTS                             ║

## CELL 2 — CONFIGURATION (EDIT THIS)


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in [
    "yfinance", "pandas-ta", "nsepy", "requests",
    "beautifulsoup4", "schedule", "pytz", "lxml"
]:
    install(pkg)

import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
import requests
import json
import os
import re
import time
import schedule
import pytz
import warnings
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from IPython.display import display, HTML, Javascript

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

IST = pytz.timezone("Asia/Kolkata")
print("✓ All dependencies loaded.")

## CELL 3 — SCANNER ENGINE


In [ ]:
# ║  CELL 2 — CONFIGURATION (EDIT THIS)                    ║

## CELL 4 — MANUAL RUN (run this anytime)


In [ ]:
CONFIG = {
    # ── Capital & Risk ──────────────────────────────────────
    "capital":              500000,     # Your total trading capital in ₹
    "risk_per_trade_pct":   2.0,        # % of capital risked per trade
    "max_open_trades":      6,          # Max concurrent positions
    "max_portfolio_risk":   10.0,       # Max total % capital at risk at once

    # ── Scanner Output ──────────────────────────────────────
    "top_n_longs":          5,          # How many buy picks to show
    "top_n_shorts":         5,          # How many short picks to show
    "output_dir":           "/content", # Where HTML report is saved

    # ── Technical Thresholds ────────────────────────────────
    "min_momentum_pct":     20.0,       # Min % move in prior leg (1 month)
    "min_daily_candle_pct": 6.5,        # OR single big daily candle
    "vcp_depth_min":        8.0,        # VCP pullback min % from peak
    "vcp_depth_max":        25.0,       # VCP pullback max % from peak
    "volume_dry_pct":       50.0,       # Pullback vol must be < X% of leg avg vol
    "min_score_long":       60,         # Min score (0-100) to qualify as long
    "min_score_short":      60,         # Min score (0-100) to qualify as short

    # ── Fundamental Filters (Longs) ─────────────────────────
    "min_market_cap_cr":    500,        # ₹ Crores
    "min_avg_volume":       300000,     # Shares/day (20-day avg)
    "max_debt_equity":      2.5,
    "min_roe":              10.0,       # %
    "min_promoter_holding": 30.0,       # %
    "max_pledged_pct":      15.0,       # % of promoter shares pledged
    "min_revenue_growth":   8.0,        # % YoY
    "min_eps_growth":       10.0,       # % YoY
    "min_delivery_pct":     35.0,       # % delivery-based trades

    # ── Fundamental Filters (Shorts — F&O only) ─────────────
    "max_lot_value_lakh":   15.0,       # Max lot value in ₹ lakhs for options
    "min_oi_growth_short":  0.0,        # OI must be flat or rising on bounce

    # ── Market Regime ────────────────────────────────────────
    "regime_index":         "^NSEI",    # Nifty 50 Yahoo ticker
    "smallcap_index":       "^CNXSC",   # Nifty Smallcap 100
    "roc_bull_min":         0,
    "roc_bull_max":         45,
    "vix_long_suppress":    25,         # Suppress longs above this VIX

    # ── Exit Framework ───────────────────────────────────────
    "partial_exit_1_r":     2.0,        # Sell 25% at 2R
    "partial_exit_1_pct":   25,
    "partial_exit_2_pct_gain": 20.0,   # Sell 25% at 20% gain
    "partial_exit_2_pct":   25,
    "trail_ema_weekly":     20,         # Weekly EMA for stage 2 trail
    "trail_sma_weekly":     30,         # 30-week SMA for multibagger trail
    "short_fixed_target_r": 2.0,        # Shorts: fixed 2R target
}

# ── NSE Universe (F&O enabled — ~180 stocks, Jan 2025 list) ─
# Full list pulled dynamically below; this is a fallback subset
FNO_STOCKS_FALLBACK = [
    "RELIANCE","TCS","HDFCBANK","INFY","ICICIBANK","HINDUNILVR",
    "SBIN","BHARTIARTL","KOTAKBANK","ITC","LT","AXISBANK",
    "ASIANPAINT","MARUTI","SUNPHARMA","TITAN","NESTLEIND","WIPRO",
    "ULTRACEMCO","POWERGRID","NTPC","TECHM","HCLTECH","BAJFINANCE",
    "BAJAJFINSV","DIVISLAB","DRREDDY","CIPLA","EICHERMOT","HEROMOTOCO",
    "ADANIPORTS","COALINDIA","BPCL","ONGC","TATASTEEL","HINDALCO",
    "JSWSTEEL","TATACONSUM","BRITANNIA","PIDILITIND","DABUR","MARICO",
    "GODREJCP","COLPAL","BERGEPAINT","HAVELLS","VOLTAS","WHIRLPOOL",
    "ESCORTS","ASHOKLEY","BALKRISIND","APOLLOTYRE","MRF","CEAT",
    "MOTHERSON","BOSCHLTD","EXIDEIND","AMBUJACEM","ACCLTD","SHREECEM",
    "GRASIM","INDUSINDBK","FEDERALBNK","BANDHANBNK","IDFCFIRSTB",
    "PNB","BANKBARODA","CANBK","UNIONBANK","AUBANK","RBLBANK",
    "MUTHOOTFIN","CHOLAFIN","BAJAJHLDNG","M&MFIN","LICHSGFIN",
    "ABCAPITAL","RECLTD","PFC","IRFC","HUDCO","IREDA",
    "HDFCLIFE","SBILIFE","ICICIGI","NIACL","STARHEALTH",
    "ZOMATO","PAYTM","NYKAA","POLICYBZR","DELHIVERY","MAPMYINDIA",
    "TATACOMM","MTNL","IDEA","INDIAMART","JUSTDIAL","INFO EDGE",
    "PERSISTENT","COFORGE","LTIM","MPHASIS","HEXAWARE","NIITTECH",
    "HAL","BEL","BHEL","BEML","CONCOR","IRCTC","RVNL",
    "ADANIENT","ADANIGREEN","ADANITRANS","ADANIPOWER","AWL",
    "TATAMOTORS","TATAPOWER","TATACHEM","TRENT","VOLTAS",
    "VEDL","NMDC","SAIL","MOIL","NATIONALUM",
    "OFSS","MCDOWELL-N","RADICO","GLOBUSSPR","UBL",
    "APOLLOHOSP","FORTIS","MAXHEALTH","MEDANTA","NARAYANA",
    "JUBLFOOD","WESTLIFE","DEVYANI","SAPPHIRE","BARBEQUE",
    "PAGEIND","ABFRL","TATACLIQ","SHOPERSTOP","VMART",
    "DMART","FRETAIL","SPENCERS","ZYDUSLIFE","TORNTPHARM",
    "AUROPHARMA","LUPIN","BIOCON","GRANULES","LAURUS",
    "PIIND","RALLIS","SUMICHEM","BAYER","SHARDACROP",
]

print(f"✓ Configuration loaded. Capital: ₹{CONFIG['capital']:,}")

## CELL 5 — AUTO SCHEDULER (run once, leave running)


In [ ]:
# ║  CELL 3 — SCANNER ENGINE                               ║

## CELL 6 — KEEP-ALIVE (run alongside Cell 5)


In [ ]:
# ── 3.1  DATA LAYER ─────────────────────────────────────────

def get_nse_fno_list():
    """Pull live F&O eligible stocks from NSE website."""
    try:
        url = "https://www.nseindia.com/api/equity-stockIndices?index=SECURITIES%20IN%20F%26O"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
            "Accept": "application/json",
            "Referer": "https://www.nseindia.com/",
        }
        session = requests.Session()
        session.get("https://www.nseindia.com", headers=headers, timeout=10)
        resp = session.get(url, headers=headers, timeout=15)
        data = resp.json()
        symbols = [d["symbol"] for d in data.get("data", [])]
        if len(symbols) > 50:
            print(f"✓ Live F&O list: {len(symbols)} stocks")
            return symbols
    except Exception as e:
        print(f"⚠ NSE F&O fetch failed ({e}), using fallback list.")
    return FNO_STOCKS_FALLBACK


def get_nse_fno_ban_list():
    """Stocks currently in F&O ban period — cannot open fresh positions."""
    try:
        url = "https://www.nseindia.com/api/live-analysis-oi-spurts-underlyings"
        headers = {"User-Agent": "Mozilla/5.0", "Referer": "https://www.nseindia.com/"}
        session = requests.Session()
        session.get("https://www.nseindia.com", headers=headers, timeout=10)
        resp = session.get(url, headers=headers, timeout=15)
        # Ban list is separate endpoint
        ban_url = "https://www.nseindia.com/api/fo-underlyings-derivatives?type=banList"
        resp2 = session.get(ban_url, headers=headers, timeout=15)
        banned = [d.get("symbol","") for d in resp2.json().get("data", [])]
        print(f"✓ F&O ban list: {len(banned)} stocks banned today")
        return set(banned)
    except:
        print("⚠ Could not fetch ban list, proceeding without it.")
        return set()


def fetch_price_data(symbol, period="2y", interval="1d"):
    """
    Fetch OHLCV from yfinance. Appends .NS for NSE.
    Returns clean DataFrame or None on failure.
    """
    ticker = symbol + ".NS" if not symbol.endswith(".NS") else symbol
    try:
        df = yf.download(ticker, period=period, interval=interval,
                         progress=False, auto_adjust=True)
        if df is None or len(df) < 60:
            return None
        df.columns = [c.lower() for c in df.columns]
        df = df[["open","high","low","close","volume"]].copy()
        df.dropna(inplace=True)
        return df
    except:
        return None


def fetch_weekly_data(symbol):
    """Fetch weekly OHLCV for weekly EMA/SMA trail calculations."""
    ticker = symbol + ".NS" if not symbol.endswith(".NS") else symbol
    try:
        df = yf.download(ticker, period="3y", interval="1wk",
                         progress=False, auto_adjust=True)
        if df is None or len(df) < 30:
            return None
        df.columns = [c.lower() for c in df.columns]
        df = df[["open","high","low","close","volume"]].copy()
        df.dropna(inplace=True)
        return df
    except:
        return None


def fetch_fundamentals(symbol):
    """
    Pull key fundamentals from yfinance Ticker.info.
    Returns dict with all required fundamental fields.
    Falls back gracefully — missing data returns None (not error).
    """
    ticker = symbol + ".NS" if not symbol.endswith(".NS") else symbol
    try:
        info = yf.Ticker(ticker).info
        mcap_cr = (info.get("marketCap") or 0) / 1e7

        # Revenue growth: use trailing revenue vs prior year
        rev_growth = None
        if info.get("revenueGrowth") is not None:
            rev_growth = info["revenueGrowth"] * 100

        # EPS growth
        eps_growth = None
        if info.get("earningsGrowth") is not None:
            eps_growth = info["earningsGrowth"] * 100

        # FCF
        fcf = info.get("freeCashflow")
        fcf_positive = (fcf is not None and fcf > 0)

        return {
            "market_cap_cr":     mcap_cr,
            "debt_equity":       info.get("debtToEquity") or 0,
            "roe":               (info.get("returnOnEquity") or 0) * 100,
            "revenue_growth":    rev_growth,
            "eps_growth":        eps_growth,
            "promoter_holding":  None,          # Not in yfinance; checked separately
            "pledged_pct":       None,           # Not in yfinance; checked separately
            "fcf_positive":      fcf_positive,
            "sector":            info.get("sector", "Unknown"),
            "industry":          info.get("industry", "Unknown"),
            "name":              info.get("longName", symbol),
            "current_price":     info.get("currentPrice") or info.get("regularMarketPrice"),
            "52w_high":          info.get("fiftyTwoWeekHigh"),
            "52w_low":           info.get("fiftyTwoWeekLow"),
            "avg_volume":        info.get("averageVolume") or 0,
            "beta":              info.get("beta"),
            "lot_size":          info.get("lotSize"),    # May not be in info; handled below
        }
    except Exception as e:
        return None


def get_lot_sizes():
    """Fetch F&O lot sizes from NSE."""
    try:
        url = "https://www.nseindia.com/api/option-chain-equities?symbol=NIFTY"
        # We approximate lot size from yfinance or use a hardcoded reference
        # In production: maintain a CSV of lot sizes updated monthly
        return {}
    except:
        return {}


# ── 3.2  MARKET REGIME ──────────────────────────────────────

def calculate_regime():
    """
    Determine Bull / Bear / Neutral market regime.
    Returns dict with regime label and supporting data.
    """
    print("\n── Calculating market regime...")

    # Nifty 50
    nifty = fetch_price_data("^NSEI", period="2y", interval="1d")
    if nifty is None:
        nifty = fetch_price_data("NIFTYBEES.NS", period="2y", interval="1d")

    # India VIX
    try:
        vix_df = yf.download("^INDIAVIX", period="30d", interval="1d",
                             progress=False, auto_adjust=True)
        vix_val = float(vix_df["Close"].iloc[-1]) if vix_df is not None and len(vix_df) > 0 else 15.0
    except:
        vix_val = 15.0

    if nifty is None or len(nifty) < 50:
        return {"regime": "UNKNOWN", "vix": vix_val, "roc_18m": 0,
                "nifty_vs_ema10": False, "nifty_vs_ema20": False,
                "description": "Could not fetch Nifty data"}

    close = nifty["close"]

    # EMAs
    ema10  = ta.ema(close, length=10).iloc[-1]
    ema20  = ta.ema(close, length=20).iloc[-1]
    cur    = close.iloc[-1]

    above_ema10 = cur > ema10
    above_ema20 = cur > ema20

    # 18-month ROC (approx 378 trading days)
    lookback = min(378, len(close) - 1)
    roc_18m = ((cur - close.iloc[-lookback]) / close.iloc[-lookback]) * 100

    # Smallcap 100 confirmation
    sc = fetch_price_data("^CNXSC", period="60d", interval="1d")
    sc_bull = False
    if sc is not None and len(sc) > 25:
        sc_close = sc["close"]
        sc_ema10 = ta.ema(sc_close, length=10).iloc[-1]
        sc_ema20 = ta.ema(sc_close, length=20).iloc[-1]
        sc_bull = sc_close.iloc[-1] > sc_ema10 and sc_close.iloc[-1] > sc_ema20

    # Regime logic
    bull_conditions = (
        above_ema10 and above_ema20 and
        CONFIG["roc_bull_min"] < roc_18m < CONFIG["roc_bull_max"]
    )
    bear_conditions = (
        not above_ema10 and not above_ema20 and roc_18m < 0
    )

    if bull_conditions:
        regime = "BULL"
        if vix_val > CONFIG["vix_long_suppress"]:
            regime = "BULL_HIGH_VIX"  # Longs OK but reduce size
    elif bear_conditions:
        regime = "BEAR"
    else:
        regime = "NEUTRAL"

    result = {
        "regime":          regime,
        "vix":             round(vix_val, 2),
        "roc_18m":         round(roc_18m, 2),
        "nifty_current":   round(cur, 2),
        "nifty_ema10":     round(ema10, 2),
        "nifty_ema20":     round(ema20, 2),
        "nifty_vs_ema10":  above_ema10,
        "nifty_vs_ema20":  above_ema20,
        "smallcap_bull":   sc_bull,
        "description":     f"Nifty {cur:.0f} | EMA10 {ema10:.0f} | EMA20 {ema20:.0f} | ROC18m {roc_18m:.1f}% | VIX {vix_val:.1f}",
    }

    print(f"   Regime: {regime} | {result['description']}")
    return result


# ── 3.3  TECHNICAL INDICATORS ───────────────────────────────

def compute_indicators(df):
    """
    Add all required technical indicators to daily DataFrame.
    Uses pandas_ta — fully vectorized, no loops.
    """
    c = df["close"]
    h = df["high"]
    l = df["low"]
    v = df["volume"]

    df["ema10"]   = ta.ema(c, length=10)
    df["ema20"]   = ta.ema(c, length=20)
    df["ema50"]   = ta.ema(c, length=50)
    df["ema200"]  = ta.ema(c, length=200)
    df["sma150"]  = ta.sma(c, length=150)   # 30-week proxy on daily
    df["atr14"]   = ta.atr(h, l, c, length=14)
    df["vol20avg"]= v.rolling(20).mean()
    df["vol50avg"]= v.rolling(50).mean()

    # Price-volume relationship
    df["vol_ratio"] = v / df["vol20avg"]

    # Rate of change
    df["roc20"]   = ta.roc(c, length=20)    # 1-month momentum
    df["roc5"]    = ta.roc(c, length=5)     # Weekly momentum

    # Relative strength vs Nifty (computed separately in scanner)
    return df


def compute_weekly_indicators(wdf):
    """Add weekly EMAs and SMAs for trail management."""
    if wdf is None or len(wdf) < 30:
        return None
    c = wdf["close"]
    wdf["ema20w"]  = ta.ema(c, length=20)
    wdf["sma30w"]  = ta.sma(c, length=30)
    return wdf


# ── 3.4  PATTERN DETECTION ──────────────────────────────────

def detect_strong_leg(df, lookback=22):
    """
    Identify the most recent strong momentum leg.
    Returns dict: found (bool), move_pct, leg_start_idx, leg_end_idx,
                  leg_high, avg_volume_during_leg
    """
    if len(df) < lookback + 5:
        return {"found": False}

    recent = df.iloc[-(lookback + 20):]
    close  = recent["close"].values
    volume = recent["volume"].values
    high   = recent["high"].values

    best = {"found": False, "move_pct": 0}

    # Scan for best 20-day window with strong directional move
    for i in range(len(close) - lookback):
        window_close = close[i:i+lookback]
        window_vol   = volume[i:i+lookback]
        window_high  = high[i:i+lookback]

        move = (window_close[-1] - window_close[0]) / window_close[0] * 100
        avg_vol = window_vol.mean()

        # Check for single big daily candle
        daily_moves = np.diff(window_close) / window_close[:-1] * 100
        max_daily   = daily_moves.max() if len(daily_moves) > 0 else 0

        if move >= CONFIG["min_momentum_pct"] or max_daily >= CONFIG["min_daily_candle_pct"]:
            if move > best["move_pct"]:
                best = {
                    "found":           True,
                    "move_pct":        round(move, 2),
                    "max_daily_pct":   round(max_daily, 2),
                    "leg_high":        window_high.max(),
                    "leg_avg_volume":  avg_vol,
                    "leg_end_price":   window_close[-1],
                    "leg_start_idx":   i,
                    "leg_end_idx":     i + lookback,
                }

    return best


def detect_vcp(df, leg):
    """
    Validate Volatility Contraction Pattern after the momentum leg.
    Checks: pullback depth, volume dry-up, EMA support.
    Returns scoring dict.
    """
    if not leg["found"] or len(df) < 10:
        return {"valid": False, "score": 0, "details": {}}

    # Current pullback from leg high
    leg_high    = leg["leg_high"]
    cur_close   = df["close"].iloc[-1]
    pullback_pct = (leg_high - cur_close) / leg_high * 100

    valid_depth = CONFIG["vcp_depth_min"] <= pullback_pct <= CONFIG["vcp_depth_max"]

    # Volume contraction: last 10 bars vs leg avg volume
    recent_vol_avg = df["volume"].iloc[-10:].mean()
    leg_vol_avg    = leg["leg_avg_volume"]
    vol_ratio      = (recent_vol_avg / leg_vol_avg * 100) if leg_vol_avg > 0 else 100
    vol_contracted = vol_ratio < CONFIG["volume_dry_pct"]

    # EMA support: price above or touching EMA10/20
    ema10 = df["ema10"].iloc[-1]
    ema20 = df["ema20"].iloc[-1]
    ema50 = df["ema50"].iloc[-1]

    above_ema10 = cur_close >= ema10 * 0.99  # 1% tolerance
    above_ema20 = cur_close >= ema20 * 0.99
    above_ema50 = cur_close >= ema50 * 0.99

    # EMA200 for Stage 2 confirmation
    ema200 = df["ema200"].iloc[-1] if not pd.isna(df["ema200"].iloc[-1]) else 0
    stage2 = cur_close > ema200 and ema50 > ema200 if ema200 > 0 else False

    details = {
        "pullback_pct":  round(pullback_pct, 2),
        "vol_ratio_pct": round(vol_ratio, 2),
        "above_ema10":   above_ema10,
        "above_ema20":   above_ema20,
        "above_ema50":   above_ema50,
        "stage2":        stage2,
        "ema10":         round(ema10, 2),
        "ema20":         round(ema20, 2),
        "ema50":         round(ema50, 2),
        "ema200":        round(ema200, 2) if ema200 > 0 else None,
    }

    valid = valid_depth and vol_contracted and above_ema20

    return {"valid": valid, "details": details}


def detect_inside_bar_or_mother_bar(df, lookback=5):
    """
    Detect inside bar (IB) or mother bar (MB) in recent price action.
    Inside bar: current bar entirely within prior bar's range.
    Mother bar: large-range bar followed by smaller bars.
    Returns: type (IB/MB/None), entry_trigger (breach level), bar_idx
    """
    if len(df) < lookback + 2:
        return {"type": None, "entry": None}

    recent = df.iloc[-lookback:]
    highs  = recent["high"].values
    lows   = recent["low"].values

    # Inside bar: last bar within second-to-last
    for i in range(len(recent)-1, 0, -1):
        if highs[i] < highs[i-1] and lows[i] > lows[i-1]:
            return {
                "type":  "IB",
                "entry": round(highs[i-1], 2),  # Entry on breach of mother bar high
                "low":   round(lows[i-1], 2),
            }

    # Mother bar: one bar with range > 1.5× average of surrounding bars
    avg_range = (recent["high"] - recent["low"]).mean()
    for i in range(len(recent)-3, len(recent)-1):
        bar_range = highs[i] - lows[i]
        if bar_range > 1.5 * avg_range:
            # Subsequent bars must be smaller
            if all((highs[j] - lows[j]) < bar_range * 0.7 for j in range(i+1, len(recent))):
                return {
                    "type":  "MB",
                    "entry": round(highs[i], 2),
                    "low":   round(lows[i], 2),
                }

    return {"type": None, "entry": None}


def detect_color_change(df):
    """
    Color change: green candle after ≥2 red pullback candles, above EMA50.
    """
    if len(df) < 4:
        return False
    closes = df["close"].values
    opens  = df["open"].values
    ema50  = df["ema50"].iloc[-1]

    last_green = closes[-1] > opens[-1]
    two_prior_red = (closes[-3] < opens[-3]) and (closes[-2] < opens[-2])
    above_ema50 = closes[-1] > ema50

    return last_green and two_prior_red and above_ema50


def detect_short_setup(df):
    """
    Detect Stage 4 breakdown / double top / weak bounce for short entries.
    Returns scoring dict.
    """
    if len(df) < 60:
        return {"valid": False, "score": 0, "details": {}}

    close  = df["close"]
    high   = df["high"]
    ema50  = df["ema50"].iloc[-1]
    ema200 = df["ema200"].iloc[-1] if not pd.isna(df["ema200"].iloc[-1]) else None
    cur    = close.iloc[-1]

    below_ema50  = cur < ema50
    below_ema200 = (cur < ema200) if ema200 else False
    stage4       = below_ema50 and below_ema200

    # Lower highs and lower lows (last 20 bars)
    recent_h = high.iloc[-20:].values
    recent_l = df["low"].iloc[-20:].values
    lower_highs = all(recent_h[i] <= recent_h[i-2] for i in range(2, len(recent_h), 2))
    lower_lows  = all(recent_l[i] <= recent_l[i-2] for i in range(2, len(recent_l), 2))
    downtrend = lower_highs and lower_lows

    # Double top: two peaks within 3% of each other in last 40 bars
    recent40_h = high.iloc[-40:].values
    peak1_idx  = np.argmax(recent40_h[:20])
    peak2_idx  = np.argmax(recent40_h[20:]) + 20
    peak1, peak2 = recent40_h[peak1_idx], recent40_h[peak2_idx]
    double_top = abs(peak1 - peak2) / max(peak1, peak2) < 0.03 if max(peak1, peak2) > 0 else False

    # Weak bounce: bounce < 30% of prior down move, low volume
    down_move   = (high.iloc[-40] - close.iloc[-10]) if len(df) >= 40 else 0
    bounce      = (close.iloc[-1] - close.iloc[-10]) if len(df) >= 10 else 0
    weak_bounce = (bounce / down_move < 0.30) if down_move > 0 else False

    # Volume weak on bounce
    vol_on_bounce = df["volume"].iloc[-5:].mean()
    vol_baseline  = df["vol20avg"].iloc[-1]
    low_vol_bounce = vol_on_bounce < vol_baseline * 0.8 if vol_baseline > 0 else False

    # Entry trigger: red candle after failed test of resistance
    recent_close = close.iloc[-3:].values
    recent_open  = df["open"].iloc[-3:].values
    red_confirm  = recent_close[-1] < recent_open[-1]

    score = 0
    if stage4:         score += 25
    if downtrend:      score += 20
    if double_top:     score += 20
    if weak_bounce:    score += 15
    if low_vol_bounce: score += 10
    if red_confirm:    score += 10

    details = {
        "below_ema50":   below_ema50,
        "below_ema200":  below_ema200,
        "stage4":        stage4,
        "downtrend":     downtrend,
        "double_top":    double_top,
        "weak_bounce":   weak_bounce,
        "low_vol_bounce":low_vol_bounce,
        "red_confirm":   red_confirm,
        "ema50":         round(ema50, 2),
        "ema200":        round(ema200, 2) if ema200 else None,
    }

    valid = stage4 and (downtrend or double_top) and score >= CONFIG["min_score_short"]

    return {"valid": valid, "score": score, "details": details}


# ── 3.5  LONG SIGNAL SCORING ────────────────────────────────

def score_long_setup(df, leg, vcp, bar_pattern, color_change, fundamentals):
    """
    Composite long signal score 0–100.
    Returns score (int) and breakdown dict.
    """
    score = 0
    breakdown = {}

    # ── Technical (70 pts) ──
    # Stage 2: above EMA50 and EMA200
    stage2 = vcp["details"].get("stage2", False)
    pts = 20 if stage2 else (10 if vcp["details"].get("above_ema50") else 0)
    score += pts; breakdown["stage2"] = pts

    # Momentum leg
    pts = 20 if leg.get("move_pct", 0) >= CONFIG["min_momentum_pct"] else (
          12 if leg.get("max_daily_pct", 0) >= CONFIG["min_daily_candle_pct"] else 0)
    score += pts; breakdown["momentum_leg"] = pts

    # VCP depth (tighter is better)
    pd_val = vcp["details"].get("pullback_pct", 99)
    pts = 10 if 8 <= pd_val <= 15 else (7 if 15 < pd_val <= 20 else (4 if vcp["valid"] else 0))
    score += pts; breakdown["vcp_depth"] = pts

    # Volume dry-up
    vr = vcp["details"].get("vol_ratio_pct", 100)
    pts = 10 if vr < 35 else (7 if vr < 50 else (3 if vr < 70 else 0))
    score += pts; breakdown["vol_dry"] = pts

    # Entry pattern
    pts = 10 if bar_pattern["type"] in ["IB","MB"] else 0
    score += pts; breakdown["bar_pattern"] = pts

    # Color change confirmation
    pts = 5 if color_change else 0
    score += pts; breakdown["color_change"] = pts

    # ── Fundamental (30 pts) ──
    if fundamentals:
        # Revenue growth
        rg = fundamentals.get("revenue_growth")
        pts = 8 if (rg and rg >= 20) else (5 if (rg and rg >= CONFIG["min_revenue_growth"]) else 0)
        score += pts; breakdown["rev_growth"] = pts

        # EPS growth
        eg = fundamentals.get("eps_growth")
        pts = 8 if (eg and eg >= 25) else (5 if (eg and eg >= CONFIG["min_eps_growth"]) else 0)
        score += pts; breakdown["eps_growth"] = pts

        # ROE
        roe = fundamentals.get("roe", 0)
        pts = 7 if roe >= 20 else (4 if roe >= CONFIG["min_roe"] else 0)
        score += pts; breakdown["roe"] = pts

        # FCF
        pts = 7 if fundamentals.get("fcf_positive") else 0
        score += pts; breakdown["fcf"] = pts

    return min(score, 100), breakdown


# ── 3.6  RISK ENGINE ────────────────────────────────────────

def calculate_risk_levels(df, fundamentals, direction="long"):
    """
    Calculate entry, SL, partial exits, trail levels, position size.
    Multibagger trail for longs — fixed 2R for shorts.
    """
    if df is None or len(df) < 20:
        return None

    cur     = df["close"].iloc[-1]
    atr     = df["atr14"].iloc[-1]
    ema10   = df["ema10"].iloc[-1]
    ema20   = df["ema20"].iloc[-1]
    sma150  = df["sma150"].iloc[-1] if not pd.isna(df["sma150"].iloc[-1]) else None

    capital   = CONFIG["capital"]
    risk_pct  = CONFIG["risk_per_trade_pct"] / 100
    risk_amt  = capital * risk_pct

    if direction == "long":
        # Entry zone: current close to +0.5%
        entry      = round(cur * 1.002, 2)     # slight buffer above close
        entry_high = round(cur * 1.008, 2)     # don't chase above this

        # SL: strictest of 2.5%, PDL, or 1.5×ATR
        pdl         = df["low"].iloc[-1]
        sl_pct      = round(entry * (1 - 0.025), 2)
        sl_atr      = round(entry - 1.5 * atr, 2)
        sl_pdl      = round(pdl * 0.995, 2)
        stop_loss   = round(max(sl_pct, sl_atr, sl_pdl), 2)   # highest = tightest

        risk_per_share   = entry - stop_loss
        if risk_per_share <= 0:
            return None

        position_size    = int(risk_amt / risk_per_share)
        position_value   = round(position_size * entry, 2)

        # Exit levels
        be_level    = entry                              # Breakeven at +2R → move SL here
        partial1_px = round(entry + 2 * risk_per_share, 2)   # Sell 25% at 2R
        partial2_px = round(entry * 1.20, 2)                  # Sell 25% at +20%

        # Trail levels (from weekly data — approximated from daily)
        trail_ema20w = round(ema20 * 0.99, 2)            # Weekly EMA20 approx
        trail_sma30w = round(sma150, 2) if sma150 else None  # 30W SMA = 150-day SMA

        return {
            "direction":       "LONG",
            "entry":           entry,
            "entry_high":      entry_high,
            "stop_loss":       stop_loss,
            "sl_method":       "max(2.5%, 1.5×ATR, PDL)",
            "risk_per_share":  round(risk_per_share, 2),
            "position_size":   position_size,
            "position_value":  position_value,
            "risk_amount":     round(risk_amt, 2),
            "breakeven_at":    be_level,
            "partial1_price":  partial1_px,
            "partial1_qty_pct":CONFIG["partial_exit_1_pct"],
            "partial2_price":  partial2_px,
            "partial2_qty_pct":CONFIG["partial_exit_2_pct"],
            "trail_stage2":    trail_ema20w,
            "trail_multibagger":trail_sma30w,
            "trail_note":      "After +40%: trail 30W SMA. Exit on weekly close below.",
        }

    else:  # SHORT
        entry     = round(cur * 0.998, 2)
        stop_loss = round(cur * 1.025, 2)    # 2.5% above entry
        sl_atr    = round(entry + 1.5 * atr, 2)
        stop_loss = round(min(stop_loss, sl_atr), 2)   # tightest

        risk_per_share = stop_loss - entry
        if risk_per_share <= 0:
            return None

        position_size  = int(risk_amt / risk_per_share)
        target_1       = round(entry - 2 * risk_per_share, 2)   # 2R target
        target_2       = round(entry - 3 * risk_per_share, 2)   # 3R stretch

        return {
            "direction":      "SHORT",
            "entry":          entry,
            "stop_loss":      stop_loss,
            "risk_per_share": round(risk_per_share, 2),
            "position_size":  position_size,
            "position_value": round(position_size * entry, 2),
            "risk_amount":    round(risk_amt, 2),
            "target_1":       target_1,
            "target_2":       target_2,
            "trail_note":     "Fixed target. Cover 50% at 2R, 50% at 3R.",
        }


# ── 3.7  SIGNAL REASON STRING ───────────────────────────────

def build_reason_string(direction, leg, vcp, bar_pattern, color_change,
                        score, breakdown, fundamentals):
    """
    Build a human-readable plain-English reason for the trade signal.
    """
    parts = []

    if direction == "long":
        if vcp["details"].get("stage2"):
            parts.append("Stage 2 uptrend confirmed (above EMA50 & EMA200)")
        if leg["found"]:
            parts.append(f"Prior leg: +{leg['move_pct']:.1f}% move")
        pd_val = vcp["details"].get("pullback_pct")
        if pd_val:
            parts.append(f"VCP pullback {pd_val:.1f}% from peak")
        vr = vcp["details"].get("vol_ratio_pct")
        if vr:
            parts.append(f"Volume dried to {vr:.0f}% of leg avg")
        if bar_pattern["type"]:
            parts.append(f"{bar_pattern['type']} pattern → entry on breach of ₹{bar_pattern['entry']}")
        if color_change:
            parts.append("Color change confirmed (green after 2 red bars, above EMA50)")
        if fundamentals:
            rg = fundamentals.get("revenue_growth")
            eg = fundamentals.get("eps_growth")
            roe = fundamentals.get("roe")
            if rg and rg > 0:   parts.append(f"Revenue +{rg:.0f}% YoY")
            if eg and eg > 0:   parts.append(f"EPS +{eg:.0f}% YoY")
            if roe and roe > 0: parts.append(f"ROE {roe:.0f}%")
            if fundamentals.get("fcf_positive"): parts.append("FCF positive")
    else:
        parts.append("Stage 4 downtrend")
        if vcp["details"].get("double_top"):    parts.append("Double top structure")
        if vcp["details"].get("downtrend"):     parts.append("Lower highs & lower lows")
        if vcp["details"].get("weak_bounce"):   parts.append("Weak bounce (<30% of prior down move)")
        if vcp["details"].get("low_vol_bounce"):parts.append("Low volume on bounce")
        if vcp["details"].get("red_confirm"):   parts.append("Red confirmation candle")

    return " | ".join(parts) if parts else "Setup detected"


def estimate_probability(score, regime, direction):
    """
    Transparent probability estimate based on score and regime alignment.
    NOT ML — rule-weighted. Will be calibrated after 3 months of logging.
    """
    base = score * 0.6   # score contributes 60% of probability

    # Regime alignment bonus
    if direction == "long" and regime in ["BULL"]:
        base += 15
    elif direction == "long" and regime == "BULL_HIGH_VIX":
        base += 8
    elif direction == "short" and regime == "BEAR":
        base += 15
    elif direction in ["long","short"] and regime == "NEUTRAL":
        base -= 10

    return round(min(max(base, 20), 88), 1)  # Cap 20-88%: never overconfident


# ── 3.8  FUNDAMENTAL FILTER ─────────────────────────────────

def passes_fundamental_filter(fundamentals, direction="long"):
    """
    Returns (True/False, rejection_reason).
    """
    if fundamentals is None:
        return False, "No fundamental data"

    f = fundamentals

    if direction == "long":
        if f.get("market_cap_cr", 0) < CONFIG["min_market_cap_cr"]:
            return False, f"Mktcap ₹{f.get('market_cap_cr',0):.0f}Cr < min"
        if f.get("avg_volume", 0) < CONFIG["min_avg_volume"]:
            return False, f"Avg vol {f.get('avg_volume',0):,.0f} < min"
        de = f.get("debt_equity", 0) or 0
        if de > CONFIG["max_debt_equity"]:
            return False, f"D/E {de:.1f} too high"
        roe = f.get("roe", 0) or 0
        if roe < CONFIG["min_roe"]:
            return False, f"ROE {roe:.1f}% < min"
        rg = f.get("revenue_growth")
        if rg is not None and rg < CONFIG["min_revenue_growth"]:
            return False, f"Rev growth {rg:.1f}% < min"
    # Shorts: fundamental filter is lighter — just liquidity and F&O validity
    else:
        if f.get("avg_volume", 0) < CONFIG["min_avg_volume"] * 0.5:
            return False, "Too illiquid for derivatives"

    return True, "OK"


# ── 3.9  MAIN SCANNER ───────────────────────────────────────

def scan_universe(regime):
    """
    Core scanner. Iterates universe, applies all filters, scores, returns
    top N longs and shorts.
    """
    print("\n── Fetching F&O universe and ban list...")
    fno_universe = get_nse_fno_list()
    ban_list     = get_nse_fno_ban_list()

    long_results  = []
    short_results = []

    regime_label = regime["regime"]
    run_longs  = regime_label in ["BULL", "BULL_HIGH_VIX", "NEUTRAL"]
    run_shorts = regime_label in ["BEAR", "NEUTRAL", "BULL"]  # Always check shorts

    # Size adjustment for regime
    size_mult = 0.5 if regime_label == "BULL_HIGH_VIX" else (
                0.75 if regime_label == "NEUTRAL" else 1.0)

    print(f"   Scanning {len(fno_universe)} stocks | Regime: {regime_label} | Size mult: {size_mult}x")
    print(f"   Running longs: {run_longs} | Running shorts: {run_shorts}")

    total = len(fno_universe)
    for idx, symbol in enumerate(fno_universe):

        if idx % 20 == 0:
            print(f"   Progress: {idx}/{total}...")

        # Skip banned stocks for shorts
        if symbol in ban_list and not run_longs:
            continue

        # ── Fetch data ──
        df = fetch_price_data(symbol, period="2y", interval="1d")
        if df is None or len(df) < 100:
            continue

        df = compute_indicators(df)
        fundamentals = fetch_fundamentals(symbol)

        # ── Fundamental pre-filter ──
        if run_longs:
            fund_ok, fund_reason = passes_fundamental_filter(fundamentals, "long")
            if not fund_ok:
                continue

        # ── LONG SCAN ──
        if run_longs and symbol not in ban_list:
            leg           = detect_strong_leg(df)
            vcp_data      = detect_vcp(df, leg)
            bar_pattern   = detect_inside_bar_or_mother_bar(df)
            color_chg     = detect_color_change(df)

            if leg["found"] and vcp_data["valid"]:
                score, breakdown = score_long_setup(
                    df, leg, vcp_data, bar_pattern, color_chg, fundamentals)

                if score >= CONFIG["min_score_long"]:
                    risk = calculate_risk_levels(df, fundamentals, "long")
                    if risk:
                        risk["position_size"] = int(risk["position_size"] * size_mult)
                        reason = build_reason_string(
                            "long", leg, vcp_data, bar_pattern, color_chg,
                            score, breakdown, fundamentals)
                        prob = estimate_probability(score, regime_label, "long")

                        long_results.append({
                            "symbol":       symbol,
                            "name":         fundamentals.get("name", symbol) if fundamentals else symbol,
                            "sector":       fundamentals.get("sector","—") if fundamentals else "—",
                            "direction":    "LONG",
                            "score":        score,
                            "probability":  prob,
                            "price":        round(df["close"].iloc[-1], 2),
                            "risk":         risk,
                            "reason":       reason,
                            "breakdown":    breakdown,
                            "52w_high":     fundamentals.get("52w_high") if fundamentals else None,
                            "52w_low":      fundamentals.get("52w_low") if fundamentals else None,
                        })

        # ── SHORT SCAN (F&O stocks only) ──
        if run_shorts and symbol not in ban_list:
            fund_ok_s, _ = passes_fundamental_filter(fundamentals, "short")
            if not fund_ok_s:
                continue

            short_data = detect_short_setup(df)

            if short_data["valid"]:
                risk_s = calculate_risk_levels(df, fundamentals, "short")
                if risk_s:
                    reason_s = build_reason_string(
                        "short", {}, {"details": short_data["details"]},
                        {}, False, short_data["score"], {}, None)
                    prob_s = estimate_probability(short_data["score"], regime_label, "short")

                    short_results.append({
                        "symbol":      symbol,
                        "name":        fundamentals.get("name", symbol) if fundamentals else symbol,
                        "sector":      fundamentals.get("sector","—") if fundamentals else "—",
                        "direction":   "SHORT",
                        "score":       short_data["score"],
                        "probability": prob_s,
                        "price":       round(df["close"].iloc[-1], 2),
                        "risk":        risk_s,
                        "reason":      reason_s,
                        "breakdown":   short_data["details"],
                    })

    # Sort and take top N
    long_results.sort(key=lambda x: x["score"], reverse=True)
    short_results.sort(key=lambda x: x["score"], reverse=True)

    top_longs  = long_results[:CONFIG["top_n_longs"]]
    top_shorts = short_results[:CONFIG["top_n_shorts"]]

    print(f"\n✓ Scanner complete: {len(long_results)} long candidates, "
          f"{len(short_results)} short candidates")
    print(f"   Top {len(top_longs)} longs, Top {len(top_shorts)} shorts selected")

    return top_longs, top_shorts


# ── 3.10  DASHBOARD HTML GENERATOR ─────────────────────────

def generate_dashboard(top_longs, top_shorts, regime):
    """
    Generate a sleek, production-grade HTML dashboard.
    Saved to output_dir as scanner_YYYYMMDD.html.
    Auto-opens in Colab via IPython display.
    """
    now_ist  = datetime.now(IST)
    date_str = now_ist.strftime("%d %b %Y, %I:%M %p IST")
    fname    = f"scanner_{now_ist.strftime('%Y%m%d_%H%M')}.html"
    fpath    = os.path.join(CONFIG["output_dir"], fname)

    regime_label = regime["regime"]
    regime_color = {
        "BULL":          "#00c896",
        "BULL_HIGH_VIX": "#f0a500",
        "BEAR":          "#ff4d6d",
        "NEUTRAL":       "#a0a0b0",
        "UNKNOWN":       "#606070",
    }.get(regime_label, "#a0a0b0")

    def pct_bar(pct, color):
        return f'''<div style="background:#1a1a2e;border-radius:4px;height:6px;width:100%;margin-top:6px">
            <div style="width:{pct}%;background:{color};height:6px;border-radius:4px;
            transition:width 1s ease"></div></div>
            <span style="font-size:11px;color:#888;margin-top:3px;display:block">{pct}% probability</span>'''

    def risk_table_long(r):
        rows = [
            ("Entry zone",       f"₹{r['entry']} – ₹{r['entry_high']}"),
            ("Stop loss",        f"₹{r['stop_loss']} ({r['sl_method']})"),
            ("Risk per share",   f"₹{r['risk_per_share']}"),
            ("Position size",    f"{r['position_size']} shares (₹{r['position_value']:,.0f})"),
            ("Capital at risk",  f"₹{r['risk_amount']:,.0f}"),
            ("Breakeven SL at",  f"₹{r['breakeven_at']} (move SL here at +2R)"),
            ("Partial exit 1",   f"₹{r['partial1_price']} → sell {r['partial1_qty_pct']}%"),
            ("Partial exit 2",   f"₹{r['partial2_price']} → sell {r['partial2_qty_pct']}%"),
            ("Stage 2 trail",    f"₹{r['trail_stage2']} (Weekly EMA20)"),
            ("Multibagger trail",f"₹{r['trail_multibagger'] or 'N/A'} (30W SMA — exit on weekly close below)"),
        ]
        return "".join(f'''<tr><td style="color:#888;padding:5px 10px 5px 0;font-size:12px;
            white-space:nowrap">{k}</td>
            <td style="color:#e0e0f0;padding:5px 0;font-size:12px;font-weight:500">{v}</td></tr>'''
            for k, v in rows)

    def risk_table_short(r):
        rows = [
            ("Entry",           f"₹{r['entry']}"),
            ("Stop loss",       f"₹{r['stop_loss']}"),
            ("Risk per share",  f"₹{r['risk_per_share']}"),
            ("Position size",   f"{r['position_size']} shares (₹{r['position_value']:,.0f})"),
            ("Capital at risk", f"₹{r['risk_amount']:,.0f}"),
            ("Target 1 (2R)",   f"₹{r['target_1']}"),
            ("Target 2 (3R)",   f"₹{r['target_2']}"),
            ("Exit note",       r['trail_note']),
        ]
        return "".join(f'''<tr><td style="color:#888;padding:5px 10px 5px 0;font-size:12px;
            white-space:nowrap">{k}</td>
            <td style="color:#e0e0f0;padding:5px 0;font-size:12px;font-weight:500">{v}</td></tr>'''
            for k, v in rows)

    def card(stock, idx):
        is_long   = stock["direction"] == "LONG"
        dir_color = "#00c896" if is_long else "#ff4d6d"
        dir_label = "BUY" if is_long else "SHORT"
        dir_icon  = "↑" if is_long else "↓"
        prob      = stock["probability"]
        r         = stock["risk"]
        rt        = risk_table_long(r) if is_long else risk_table_short(r)
        score_clr = "#00c896" if stock["score"] >= 80 else (
                    "#f0a500" if stock["score"] >= 65 else "#ff4d6d")
        w52h = stock.get("52w_high")
        w52l = stock.get("52w_low")
        w52_str = f"52W: ₹{w52l:.0f} – ₹{w52h:.0f}" if w52h and w52l else ""

        return f'''
        <div style="background:linear-gradient(135deg,#12122a 0%,#1a1a3a 100%);
             border:1px solid #2a2a4a;border-radius:16px;padding:24px 28px;
             margin-bottom:20px;position:relative;overflow:hidden">
          <div style="position:absolute;top:0;right:0;width:120px;height:120px;
               background:radial-gradient(circle,{dir_color}18 0%,transparent 70%);
               pointer-events:none"></div>
          <div style="display:flex;justify-content:space-between;align-items:flex-start;
               flex-wrap:wrap;gap:12px">
            <div>
              <div style="display:flex;align-items:center;gap:10px;margin-bottom:4px">
                <span style="background:{dir_color}22;color:{dir_color};
                     border:1px solid {dir_color}44;border-radius:6px;
                     padding:3px 10px;font-size:12px;font-weight:700;letter-spacing:1px">
                  {dir_icon} {dir_label}</span>
                <span style="color:#888;font-size:12px">#{idx+1}</span>
              </div>
              <div style="font-size:22px;font-weight:800;color:#ffffff;
                   letter-spacing:-0.5px;line-height:1.2">{stock["symbol"]}</div>
              <div style="font-size:13px;color:#aaa;margin-top:2px">{stock["name"]}</div>
              <div style="font-size:11px;color:#666;margin-top:2px">
                {stock["sector"]} &nbsp;|&nbsp; {w52_str}</div>
            </div>
            <div style="text-align:right">
              <div style="font-size:28px;font-weight:800;color:#fff">
                ₹{stock["price"]:,.2f}</div>
              <div style="font-size:11px;color:#666">last close</div>
              <div style="margin-top:8px">
                <span style="font-size:13px;font-weight:700;color:{score_clr}">
                  Score {stock["score"]}/100</span>
              </div>
            </div>
          </div>
          <div style="margin:16px 0 12px">
            {pct_bar(prob, dir_color)}
          </div>
          <div style="background:#0d0d20;border-radius:10px;padding:14px 16px;
               margin-bottom:16px;border-left:3px solid {dir_color}">
            <div style="font-size:11px;color:#666;text-transform:uppercase;
                 letter-spacing:1px;margin-bottom:6px">Signal reasoning</div>
            <div style="font-size:13px;color:#c8c8e0;line-height:1.6">
              {stock["reason"]}</div>
          </div>
          <table style="width:100%;border-collapse:collapse">
            {rt}
          </table>
        </div>'''

    long_cards  = "".join(card(s, i) for i, s in enumerate(top_longs))  if top_longs  else \
                  '<p style="color:#666;text-align:center;padding:40px">No qualifying long setups today.</p>'
    short_cards = "".join(card(s, i) for i, s in enumerate(top_shorts)) if top_shorts else \
                  '<p style="color:#666;text-align:center;padding:40px">No qualifying short setups today.</p>'

    # Total portfolio risk
    total_risk = sum(s["risk"]["risk_amount"] for s in top_longs + top_shorts if s.get("risk"))
    total_risk_pct = round(total_risk / CONFIG["capital"] * 100, 1)

    html = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>NSE Swing Scanner — {date_str}</title>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{background:#08080f;color:#e0e0f0;font-family:"Space Grotesk",sans-serif;
       min-height:100vh;padding:0}}
  .mono{{font-family:"JetBrains Mono",monospace}}
  @keyframes fadeUp{{from{{opacity:0;transform:translateY(20px)}}to{{opacity:1;transform:translateY(0)}}}}
  .card-anim{{animation:fadeUp 0.5s ease both}}
  .regime-pill{{display:inline-flex;align-items:center;gap:8px;
    background:{regime_color}18;color:{regime_color};
    border:1px solid {regime_color}44;border-radius:30px;
    padding:6px 16px;font-size:13px;font-weight:600}}
  .section-title{{font-size:18px;font-weight:800;letter-spacing:-0.3px;
    padding:12px 0 16px;border-bottom:1px solid #1e1e3a;margin-bottom:20px}}
  .long-title{{color:#00c896}} .short-title{{color:#ff4d6d}}
  .stat-box{{background:#12122a;border:1px solid #2a2a4a;border-radius:12px;
    padding:16px 20px;flex:1;min-width:140px}}
  .stat-val{{font-size:24px;font-weight:800;line-height:1}}
  .stat-lbl{{font-size:11px;color:#666;margin-top:4px;text-transform:uppercase;
    letter-spacing:0.5px}}
  .divider{{height:1px;background:linear-gradient(90deg,transparent,#2a2a4a,transparent);
    margin:32px 0}}
</style>
</head>
<body>
<div style="max-width:900px;margin:0 auto;padding:32px 20px">

  <!-- Header -->
  <div style="margin-bottom:32px">
    <div style="display:flex;justify-content:space-between;align-items:flex-start;
         flex-wrap:wrap;gap:16px">
      <div>
        <div style="font-size:11px;color:#444;text-transform:uppercase;
             letter-spacing:2px;margin-bottom:8px">NSE Swing Trading Scanner</div>
        <h1 style="font-size:32px;font-weight:800;color:#fff;letter-spacing:-1px;
             line-height:1.1">Daily Watchlist</h1>
        <div style="font-size:13px;color:#555;margin-top:6px">{date_str}</div>
      </div>
      <div style="text-align:right">
        <div class="regime-pill">
          <span style="width:8px;height:8px;border-radius:50%;
               background:{regime_color};display:inline-block"></span>
          {regime_label.replace("_"," ")}
        </div>
        <div style="font-size:12px;color:#555;margin-top:8px;max-width:260px;
             text-align:right">{regime["description"]}</div>
      </div>
    </div>
  </div>

  <!-- Stats row -->
  <div style="display:flex;gap:12px;flex-wrap:wrap;margin-bottom:32px">
    <div class="stat-box">
      <div class="stat-val" style="color:#00c896">{len(top_longs)}</div>
      <div class="stat-lbl">Long setups</div>
    </div>
    <div class="stat-box">
      <div class="stat-val" style="color:#ff4d6d">{len(top_shorts)}</div>
      <div class="stat-lbl">Short setups</div>
    </div>
    <div class="stat-box">
      <div class="stat-val" style="color:#e0e0f0">₹{total_risk:,.0f}</div>
      <div class="stat-lbl">Total risk if all taken</div>
    </div>
    <div class="stat-box">
      <div class="stat-val" style="color:{"#f0a500" if total_risk_pct>8 else "#00c896"}">
        {total_risk_pct}%</div>
      <div class="stat-lbl">Portfolio risk %</div>
    </div>
    <div class="stat-box">
      <div class="stat-val mono" style="color:#a0a0f0">
        {regime["vix"]}</div>
      <div class="stat-lbl">India VIX</div>
    </div>
  </div>

  <!-- Longs -->
  <div class="section-title long-title">↑ Long Candidates — Buy Setups</div>
  <div id="longs">
    {long_cards}
  </div>

  <div class="divider"></div>

  <!-- Shorts -->
  <div class="section-title short-title">↓ Short Candidates — F&O Only</div>
  <div style="background:#ff4d6d11;border:1px solid #ff4d6d22;border-radius:10px;
       padding:10px 16px;margin-bottom:20px;font-size:12px;color:#ff8899">
    Shorts are on F&O-enabled stocks only. Verify margin requirements before entry.
    Avoid stocks currently in F&O ban period.
  </div>
  <div id="shorts">
    {short_cards}
  </div>

  <!-- Footer -->
  <div style="margin-top:48px;padding-top:24px;border-top:1px solid #1a1a2e;
       font-size:11px;color:#333;text-align:center;line-height:2">
    Generated by NSE Swing Scanner &nbsp;|&nbsp; Capital: ₹{CONFIG["capital"]:,} &nbsp;|&nbsp;
    Risk/trade: {CONFIG["risk_per_trade_pct"]}% &nbsp;|&nbsp;
    This is a scanning tool, not investment advice. All trades carry risk.
  </div>
</div>
<script>
  document.querySelectorAll('.card-anim').forEach((el,i)=>{{
    el.style.animationDelay = i*0.08+"s";
  }});
</script>
</body>
</html>'''

    with open(fpath, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"\n✓ Dashboard saved: {fpath}")
    return fpath, html


# ── 3.11  MASTER RUN FUNCTION ───────────────────────────────

def run_scanner(manual=False):
    """
    Master function — runs the full pipeline end to end.
    Call this manually any time, or let the scheduler call it at 16:00 IST.
    """
    now_ist = datetime.now(IST)
    print(f"\n{'='*60}")
    print(f"  NSE SWING SCANNER — {now_ist.strftime('%d %b %Y %H:%M IST')}")
    print(f"  Mode: {'MANUAL' if manual else 'SCHEDULED'}")
    print(f"{'='*60}")

    # Step 1: Market regime
    regime = calculate_regime()

    # Step 2: Scan universe
    top_longs, top_shorts = scan_universe(regime)

    # Step 3: Generate dashboard
    fpath, html = generate_dashboard(top_longs, top_shorts, regime)

    # Step 4: Display in Colab
    display(HTML(f'''
    <div style="background:#0a0a1a;border:1px solid #2a2a4a;border-radius:12px;
         padding:16px 20px;margin:12px 0;font-family:monospace">
      <span style="color:#00c896">✓ Scanner complete</span>
      <span style="color:#666"> | </span>
      <span style="color:#aaa">{len(top_longs)} longs, {len(top_shorts)} shorts</span>
      <span style="color:#666"> | </span>
      <a href="{fpath}" style="color:#7878ff">Open full report</a>
    </div>
    '''))

    # Inline preview of top picks
    if top_longs or top_shorts:
        display(HTML(html))

    next_run = now_ist.replace(hour=16, minute=0, second=0)
    if now_ist.hour >= 16:
        next_run += timedelta(days=1)
    print(f"\n  Next scheduled run: {next_run.strftime('%d %b %Y 16:00 IST')}")
    print(f"{'='*60}\n")

    return top_longs, top_shorts, regime


print("✓ Scanner engine loaded. Call run_scanner(manual=True) to run now.")

## Cell 7


In [ ]:
# ║  CELL 4 — MANUAL RUN (run this anytime)                ║

## Cell 8


In [ ]:
# Uncomment and run this cell to trigger the scanner immediately:
# run_scanner(manual=True)

## Cell 9


In [ ]:
# ║  CELL 5 — AUTO SCHEDULER (run once, leave running)     ║

## Cell 10


In [ ]:
def start_scheduler():
    """
    Schedules a single daily run at 16:00 IST.
    Uses a lightweight 30-second sleep loop — minimal CPU.
    The session must remain open (use Cell 6 keep-alive JS alongside this).

    How it works:
      - Converts 16:00 IST to UTC for the schedule library
      - Checks every 30 seconds if a job is due
      - If you restart Colab, just re-run this cell
    """
    # IST is UTC+5:30 → 16:00 IST = 10:30 UTC
    schedule.clear()
    schedule.every().day.at("10:30").do(run_scanner)   # UTC time

    now_ist  = datetime.now(IST)
    next_run_str = now_ist.replace(hour=16, minute=0, second=0)
    if now_ist.hour >= 16:
        next_run_str += timedelta(days=1)

    print(f"✓ Scheduler armed.")
    print(f"  Next run: {next_run_str.strftime('%d %b %Y at 16:00 IST')}")
    print(f"  Checking every 30 seconds. Session must stay open.")
    print(f"  Run Cell 6 (keep-alive JS) to prevent Colab timeout.\n")

    while True:
        schedule.run_pending()
        time.sleep(30)

# Uncomment to start:
# start_scheduler()

## Cell 11


In [ ]:
# ║  CELL 6 — KEEP-ALIVE (run alongside Cell 5)            ║

## Cell 12


In [ ]:
# This JavaScript clicks the Colab "connect" button every 55 minutes
# to prevent the session from timing out during the idle window
# between 4 PM runs.
# Run this cell ONCE after starting the scheduler.

KEEPALIVE_JS = """
function keepAlive() {
  // Click the connect/reconnect button if session is about to timeout
  var connectBtn = document.querySelector('#top-toolbar > colab-connect-button');
  if (connectBtn) {
    var btn = connectBtn.shadowRoot ? 
              connectBtn.shadowRoot.querySelector('paper-icon-button') : null;
    if (btn) btn.click();
  }
  // Also click any "Stay connected" dialog
  var dialogs = document.querySelectorAll('paper-dialog:not([aria-hidden])');
  dialogs.forEach(function(d) {
    var buttons = d.querySelectorAll('paper-button');
    buttons.forEach(function(b) {
      if (b.textContent.toLowerCase().includes('stay')) b.click();
    });
  });
  console.log('[KeepAlive] Pinged at ' + new Date().toLocaleTimeString('en-IN',
    {timeZone:'Asia/Kolkata'}));
}
// Run every 55 minutes (3,300,000 ms)
setInterval(keepAlive, 3300000);
keepAlive();  // Run immediately once
console.log('[KeepAlive] Started. Will ping every 55 minutes.');
"""

# display(Javascript(KEEPALIVE_JS))
# Uncomment the line above and run this cell to activate keep-alive.